# gpu-quantum-sim: Colab quickstart (free T4 GPU)

This notebook reproduces the GPU part of the project on a free Google Colab T4:

1. **GHZ statevector benchmark** on the `nvidia` target (cuStateVec, fp32), n = 20…30 qubits.
2. **Tests** and, optionally, the **H₂ UCCSD-VQE** sweep.
3. **Plots**, merged with the CPU results already in the repository.

All logic lives in `src/`; the cells below only call the same commands as the `Makefile`.

**Before you start:** *Runtime → Change runtime type → T4 GPU*.

## 1. Check the GPU

In [ ]:
!nvidia-smi

## 2. Get the code and the pinned dependencies (~2–3 min)

In [ ]:
!git clone https://github.com/8kk24g7bgc-coder/gpu-quantum-sim.git
%cd gpu-quantum-sim
!pip install -q -r requirements.txt

## 3. Hardware and CUDA-Q check

`cudaq_gpu_count` must be at least 1, otherwise the GPU benchmark is skipped.

In [ ]:
!python -m src.hardware

## 4. Tests

In [ ]:
!make test

## 5. GHZ benchmark on the GPU (~5–15 min)

1 warm-up + 5 timed runs per qubit count; sizes that do not fit in VRAM are skipped.
The CSV is saved after every qubit count.

In [ ]:
!make ghz GHZ_TARGETS=nvidia GHZ_TAG=gpu_t4

### Optional: CPU baseline on Colab

The reference CPU baseline in the repository comes from a cluster node (8-core Xeon E-2388G, n = 20…31).
Colab only has 2 vCPUs and ~12 GB of RAM, so a CPU run here is slow (roughly an hour up to n = 26);
uncomment to run it anyway.

In [ ]:
# !make ghz GHZ_TARGETS=qpp-cpu GHZ_TAG=cpu_colab GHZ_ARGS="--max-qubits 26"

## 6. H₂ VQE (optional, ~3 min)

Re-runs the full potential-energy-curve sweep to check that it reproduces on this machine.

In [ ]:
# !make vqe

## 7. Merge all GHZ results and regenerate every plot

In [ ]:
!make plots

In [ ]:
from IPython.display import Image, display
for name in ["ghz_benchmark", "ghz_speedup", "h2_pes", "h2_error", "vqe_convergence"]:
    try:
        display(Image(f"results/{name}.png", width=650))
    except FileNotFoundError:
        print(f"results/{name}.png not found")

## 8. Download the results

Add `ghz_benchmark_gpu_t4.csv` and its `_hardware.json` to `results/` in your repository.

In [ ]:
!zip -q -r results_colab.zip results
from google.colab import files
files.download("results_colab.zip")